# LA Studio public-media downloader

This dedicated **Colab CPU** worker downloads public HTTPS media links
with yt-dlp, then lets LA Studio copy the completed file into its local
media library. It does not run a GPU model, API Gateway, a desktop
browser, desktop yt-dlp, or browser cookies.

1. Run all cells once in this Colab runtime.
2. Copy the temporary URL and token printed by the last cell.
3. In LA Studio, choose **Download with Colab**, paste them, then press
   **Connect and check Colab downloader** before adding links.


In [ ]:
%pip install -q --upgrade --no-cache-dir "fastapi==0.115.12" "uvicorn==0.34.3" "pydantic==2.10.6" "yt-dlp>=2025.08.22"


In [ ]:
from pathlib import Path
Path('/content/la_studio_media_download_worker.py').write_text('import ipaddress\nimport os\nimport re\nimport secrets\nimport socket\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\nfrom urllib.parse import urlparse\n\nfrom fastapi import FastAPI, Header, HTTPException\nfrom fastapi.responses import FileResponse\nfrom pydantic import BaseModel, Field\n\nTOKEN = os.environ["LA_STUDIO_COLAB_MEDIA_DOWNLOAD_TOKEN"]\nMODEL_ID = "yt-dlp-media-download"\nWORKER_REVISION = "media-download-2026-08-14.1"\nRESPONSE_CONTRACT = "media-download-jobs-v1"\n# requires_cuda=False: this is an acquisition-only Colab CPU worker.\n# It never runs a local/GPU inference fallback.\nREQUIRES_CUDA = False\nDOWNLOAD_ROOT = Path("/content/la_studio_media_downloads")\nMAX_FILE_BYTES = 4 * 1024 * 1024 * 1024\nMAX_JOBS = 8\nJOBS = {}\nJOB_LOCK = threading.Lock()\nURL_PATTERN = re.compile(r"^https://[^\\s]+$", re.IGNORECASE)\n\napp = FastAPI(title="LA Studio Colab media downloader", docs_url=None,\n              redoc_url=None, openapi_url=None)\n\nclass DownloadRequest(BaseModel):\n    url: str = Field(min_length=8, max_length=4096)\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\ndef public_url_or_error(value: str) -> str:\n    raw = value.strip()\n    parsed = urlparse(raw)\n    if (not URL_PATTERN.fullmatch(raw) or parsed.scheme.lower() != "https"\n            or not parsed.hostname or parsed.username or parsed.password):\n        raise HTTPException(status_code=422, detail="only one public HTTPS media URL is accepted")\n    host = parsed.hostname.rstrip(".").lower()\n    if host == "localhost" or host.endswith(".local"):\n        raise HTTPException(status_code=422, detail="local media URLs are not accepted")\n    try:\n        addresses = {record[4][0] for record in socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)}\n    except socket.gaierror:\n        raise HTTPException(status_code=422, detail="the public media host could not be resolved")\n    if not addresses:\n        raise HTTPException(status_code=422, detail="the public media host could not be resolved")\n    for address in addresses:\n        ip = ipaddress.ip_address(address)\n        if (ip.is_private or ip.is_loopback or ip.is_link_local or ip.is_multicast\n                or ip.is_reserved or ip.is_unspecified):\n            raise HTTPException(status_code=422, detail="the media URL must resolve to a public host")\n    return raw\n\ndef safe_name(path: Path) -> str:\n    name = re.sub(r"[^A-Za-z0-9._-]+", "-", path.name).strip(".-")\n    return (name or "downloaded-media")[:180]\n\ndef job_view(job: dict) -> dict:\n    directory = Path(job["directory"])\n    received = sum(path.stat().st_size for path in directory.glob("*") if path.is_file())\n    return {\n        "state": job["state"],\n        "received_bytes": min(received, MAX_FILE_BYTES),\n        "total_bytes": job.get("total_bytes", -1),\n        "file_name": job.get("file_name", ""),\n        "detail": job.get("detail", ""),\n    }\n\ndef download_job(job_id: str, public_url: str) -> None:\n    with JOB_LOCK:\n        job = JOBS[job_id]\n        job["state"] = "downloading"\n    directory = Path(job["directory"])\n    log_path = directory / "yt-dlp.log"\n    command = [\n        sys.executable, "-m", "yt_dlp", "--no-config", "--no-playlist",\n        "--restrict-filenames", "--no-progress", "--socket-timeout", "30",\n        "--retries", "3", "--fragment-retries", "3", "--max-filesize", "4G",\n        "--output", str(directory / "%(title).100B-%(id)s.%(ext)s"), public_url,\n    ]\n    try:\n        with log_path.open("w", encoding="utf-8", errors="replace") as log:\n            result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT,\n                                    timeout=60 * 45, check=False)\n        files = [path for path in directory.iterdir()\n                 if path.is_file() and path.name != log_path.name]\n        if result.returncode != 0 or len(files) != 1:\n            raise RuntimeError("yt-dlp could not download this public media")\n        output = files[0]\n        if output.stat().st_size <= 0 or output.stat().st_size > MAX_FILE_BYTES:\n            raise RuntimeError("the downloaded media is empty or exceeds the 4 GiB limit")\n        with JOB_LOCK:\n            job["state"] = "ready"\n            job["file_path"] = str(output)\n            job["file_name"] = safe_name(output)\n            job["total_bytes"] = output.stat().st_size\n            job["detail"] = ""\n    except Exception:\n        with JOB_LOCK:\n            job["state"] = "failed"\n            job["detail"] = "Colab could not download this public media. Check that the link is public and try again."\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready", "ready": True, "device": "colab-cpu",\n        "model": MODEL_ID, "variant": "fixed", "cpu_fallback": False,\n        "worker_revision": WORKER_REVISION,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1, "device": "colab-cpu",\n        "worker_revision": WORKER_REVISION,\n        "capabilities": [{"id": "media-download", "models": [{\n            "id": MODEL_ID, "name": "Colab yt-dlp media download", "variant": "fixed",\n            "loaded": True, "device": "colab-cpu", "cpu_fallback": False,\n            "response_contract": RESPONSE_CONTRACT,\n        }]}],\n    }\n\n@app.post("/v1/media/downloads")\ndef create_download(request: DownloadRequest, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    public_url = public_url_or_error(request.url)\n    with JOB_LOCK:\n        active = sum(job["state"] in {"queued", "downloading"} for job in JOBS.values())\n        if active >= MAX_JOBS:\n            raise HTTPException(status_code=429, detail="the Colab download queue is full; wait for a job to finish")\n        job_id = secrets.token_hex(16)\n        directory = DOWNLOAD_ROOT / job_id\n        directory.mkdir(parents=True, exist_ok=False)\n        JOBS[job_id] = {"state": "queued", "directory": str(directory), "created_at": time.time()}\n    thread = threading.Thread(target=download_job, args=(job_id, public_url), daemon=True)\n    thread.start()\n    return {"job_id": job_id}\n\n@app.get("/v1/media/downloads/{job_id}")\ndef download_status(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with JOB_LOCK:\n        job = JOBS.get(job_id)\n        if job is None:\n            raise HTTPException(status_code=404, detail="download job was not found")\n        return job_view(job)\n\n@app.get("/v1/media/downloads/{job_id}/file")\ndef download_file(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with JOB_LOCK:\n        job = JOBS.get(job_id)\n        if job is None or job.get("state") != "ready":\n            raise HTTPException(status_code=409, detail="download is not ready")\n        path = Path(job.get("file_path", ""))\n        name = job.get("file_name", "downloaded-media")\n    if not path.is_file() or path.stat().st_size <= 0 or path.stat().st_size > MAX_FILE_BYTES:\n        raise HTTPException(status_code=410, detail="completed media is no longer available")\n    return FileResponse(path, media_type="application/octet-stream", filename=name,\n                        headers={"Cache-Control": "no-store"})\n', encoding='utf-8')


In [ ]:
# LA Studio worker launch contract: launch-2026-08-06.1
import json
import os
import queue
import re
import secrets
import signal
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CAPABILITY_LABEL = 'media-download'
MODEL_ID = 'yt-dlp-media-download'
PORT = 8013
TOKEN_ENV = 'LA_STUDIO_COLAB_MEDIA_DOWNLOAD_TOKEN'
URL_ENV = 'LA_STUDIO_COLAB_MEDIA_DOWNLOAD_URL'
MODEL_ENV = 'LA_STUDIO_COLAB_MEDIA_DOWNLOAD_MODEL'
WORKER_LOG = Path('/content/la_studio_media_download_worker.log')
WORKER_MODULE = 'la_studio_media_download_worker'
WORKER_PYTHON = sys.executable
WORKER_PYTHON_ISOLATED = False
WORKER_ENVIRONMENT = {}
REQUIRES_CUDA = False
STARTUP_TIMEOUT_SECONDS = 20 * 60
TUNNEL_TIMEOUT_SECONDS = 90
TOKEN = secrets.token_urlsafe(32)


def port_is_occupied(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            return True
    except OSError:
        return False


def process_cmdline(pid: int) -> str:
    """Read a Linux process command line without depending on psutil."""
    try:
        return Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(
            "utf-8", errors="replace"
        ).strip()
    except (FileNotFoundError, PermissionError, ProcessLookupError):
        return ""


def all_processes():
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        pid = int(entry.name)
        command = process_cmdline(pid)
        if command:
            yield pid, command


def listening_processes(port: int) -> dict[int, str]:
    """Return PIDs listening on a local TCP port via /proc socket ownership."""
    target_port = f"{port:04X}"
    socket_inodes = set()
    for table_name in ("/proc/net/tcp", "/proc/net/tcp6"):
        try:
            lines = Path(table_name).read_text(encoding="utf-8").splitlines()[1:]
        except FileNotFoundError:
            continue
        for line in lines:
            fields = line.split()
            if len(fields) < 10:
                continue
            local_address, state, inode = fields[1], fields[3], fields[9]
            if state == "0A" and local_address.rsplit(":", 1)[-1].upper() == target_port:
                socket_inodes.add(inode)
    if not socket_inodes:
        return {}

    listeners = {}
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        try:
            descriptors = (entry / "fd").iterdir()
        except (FileNotFoundError, PermissionError):
            continue
        for descriptor in descriptors:
            try:
                target = os.readlink(descriptor)
            except (FileNotFoundError, PermissionError, OSError):
                continue
            match = re.fullmatch(r"socket:\\[(\\d+)\\]", target)
            if match and match.group(1) in socket_inodes:
                pid = int(entry.name)
                listeners[pid] = process_cmdline(pid)
                break
    return listeners


def stop_pid(pid: int) -> None:
    if pid == os.getpid():
        return
    try:
        os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    deadline = time.monotonic() + 10
    while time.monotonic() < deadline:
        try:
            os.kill(pid, 0)
        except ProcessLookupError:
            return
        time.sleep(0.2)
    try:
        os.kill(pid, signal.SIGKILL)
    except ProcessLookupError:
        pass


def reclaim_previous_la_studio_worker() -> None:
    """Stop only an older LA Studio worker/tunnel for this exact local port.

    Re-running a Colab cell keeps child processes alive.  The previous launch
    created a new token but aborted before it could replace the old worker,
    forcing users to destroy the whole GPU runtime.  We identify ownership by
    the exact generated module name and never terminate a foreign listener.
    """
    stopped = []
    for pid, command in listening_processes(PORT).items():
        if WORKER_MODULE in command and "uvicorn" in command:
            stop_pid(pid)
            stopped.append(f"worker PID {pid}")

    endpoint = f"http://127.0.0.1:{PORT}"
    for pid, command in all_processes():
        if ("cloudflared" in command and "tunnel" in command and endpoint in command):
            stop_pid(pid)
            stopped.append(f"tunnel PID {pid}")

    deadline = time.monotonic() + 12
    while port_is_occupied(PORT) and time.monotonic() < deadline:
        time.sleep(0.2)
    if stopped:
        print("Stopped previous LA Studio " + ", ".join(stopped) + ".")

    if port_is_occupied(PORT):
        listeners = listening_processes(PORT)
        foreign_pids = sorted(listeners) or ["unknown"]
        raise RuntimeError(
            f"Port {PORT} is occupied by a process that is not the previous LA Studio "
            f"{CAPABILITY_LABEL} worker (PID(s): {', '.join(map(str, foreign_pids))}). "
            "Choose a fresh Colab runtime rather than terminating an unrelated process."
        )


def worker_log_tail() -> str:
    try:
        return WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-12000:]
    except FileNotFoundError:
        return "(worker log was not created)"


def stop_process(process) -> None:
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()


reclaim_previous_la_studio_worker()

env = os.environ.copy()
env[TOKEN_ENV] = TOKEN
env["PYTHONUNBUFFERED"] = "1"
env.update(WORKER_ENVIRONMENT)
if WORKER_PYTHON_ISOLATED:
    # Do not let Colab's global site-packages or a notebook-level PYTHONPATH
    # bleed into a dedicated worker virtual environment.
    env.pop("PYTHONPATH", None)
    env["PYTHONNOUSERSITE"] = "1"
worker = None
tunnel = None

with WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:
    worker = subprocess.Popen(
        [WORKER_PYTHON, "-m", "uvicorn", 'la_studio_media_download_worker:app', "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content",
        env=env,
        stdout=worker_output,
        stderr=subprocess.STDOUT,
    )
    worker_kind = "exact CUDA" if REQUIRES_CUDA else "dedicated Colab CPU"
    print(f"Starting {worker_kind} {CAPABILITY_LABEL} worker.")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    last_error = "worker has not answered /health yet"
    next_report = time.monotonic()
    while time.monotonic() < deadline:
        exit_code = worker.poll()
        if exit_code is not None:
            raise RuntimeError(
                f"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\n\n"
                "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
            )
        try:
            request = urllib.request.Request(
                f"http://127.0.0.1:{PORT}/health",
                headers={"Authorization": "Bearer " + TOKEN},
            )
            with urllib.request.urlopen(request, timeout=10) as response:
                health = json.loads(response.read().decode("utf-8"))
            if (response.status == 200
                    and health.get("ready") is True
                    and str(health.get("device", "")).lower()
                        == ("cuda" if REQUIRES_CUDA else "colab-cpu")
                    and str(health.get("model", "")).strip().lower() == MODEL_ID
                    and health.get("cpu_fallback") is False):
                print(worker_kind.title() + " worker is ready:", health)
                break
            last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)
        except urllib.error.HTTPError as error:
            last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]
        except Exception as error:
            last_error = f"/health is not ready: {type(error).__name__}: {error}"
        if time.monotonic() >= next_report:
            print(f"Waiting for the {worker_kind} worker...", last_error)
            next_report = time.monotonic() + 30
        time.sleep(2)
    else:
        stop_process(worker)
        raise RuntimeError(
            f"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within "
            f"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\n\n"
            "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
        )


def cloudflared_ready() -> bool:
    try:
        return subprocess.run(
            ["cloudflared", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    except OSError:
        return False


def ensure_cloudflared() -> None:
    if cloudflared_ready():
        return
    package_path = "/content/la-studio-cloudflared.deb"
    download = subprocess.run(
        [
            "curl", "--fail", "--location", "--retry", "4", "--retry-all-errors",
            "--output", package_path,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if download.returncode != 0:
        detail = download.stdout[-1200:].strip() or "no download output"
        raise RuntimeError("Could not download cloudflared: " + detail)
    install = subprocess.run(
        ["dpkg", "-i", package_path], text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False,
    )
    if install.returncode != 0 or not cloudflared_ready():
        detail = install.stdout[-1200:].strip() or "no installation output"
        raise RuntimeError("Could not install cloudflared: " + detail)


ensure_cloudflared()
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tunnel_lines = queue.Queue()


def collect_tunnel_output() -> None:
    assert tunnel.stdout is not None
    for line in tunnel.stdout:
        tunnel_lines.put(line)


threading.Thread(target=collect_tunnel_output, daemon=True).start()
public_url = ""
recent_tunnel_lines = []
deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS
while time.monotonic() < deadline and not public_url:
    if tunnel.poll() is not None:
        break
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    recent_tunnel_lines.append(line.rstrip())
    recent_tunnel_lines = recent_tunnel_lines[-10:]
    print(line, end="")
    match = re.search(r"https://[^\s\"']+\.trycloudflare\.com", line)
    if match:
        # The desktop Check Colab action is the authoritative public endpoint,
        # bearer-token, capability, and exact-model verification.
        public_url = match.group(0)

if not public_url:
    stop_process(tunnel)
    stop_process(worker)
    tail = "\n".join(recent_tunnel_lines) or "(no cloudflared output)"
    raise RuntimeError(
        f"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\n"
        "---- cloudflared output ----\n" + tail
    )

os.environ[URL_ENV] = public_url
os.environ[TOKEN_ENV] = TOKEN
os.environ[MODEL_ENV] = MODEL_ID
print("\nLA Studio exact-model Colab worker is ready")
print(URL_ENV + "=" + public_url)
print(TOKEN_ENV + "=" + TOKEN)
print(MODEL_ENV + "=" + MODEL_ID)
print("Click Check Colab in the matching LA Studio feature before running it.")
